# Aktivnosti 4 i 5 - modelovanje cena nekretnina

**Tema:** Sistem za predikciju cena nekretnina primenom metoda mašinskog učenja  
**Cilj notebook-a:** jasan prikaz Aktivnosti 4 i Aktivnosti 5 prema CS490 zahtevima.

Notebook se oslanja na prethodno urađenu eksplorativnu analizu i pripremu podataka iz `notebooks/analysis.ipynb`. U ovoj svesci fokus je na modelovanju:

- **Aktivnost 4:** implementacija prvog modela / algoritma i osnovni izlaz sistema.
- **Aktivnost 5:** dodatni modeli, evaluacija rezultata i trenutni status projekta.
- **Tema projekta / Nivo 1:** implementacija najmanje tri modela mašinskog učenja: Linear Regression, Random Forest i Gradient Boosting.

Baseline model se koristi samo kao referenca. Baseline se ne računa u tri zahtevana ML modela.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "nekretnine_raw.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from src.data_cleaning import build_cleaned_dataset, create_cleaning_summary
from src.evaluation import compare_regression_results, evaluate_regression_predictions
from src.features import add_model_features
from src.model_pipeline import build_model_pipeline, get_model_feature_columns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)
RANDOM_STATE = 42

## 1. Učitavanje i priprema podataka

Koristi se fajl `data/nekretnine_raw.csv`, prikupljen web scraping-om sa portala `nekretnine.rs`. Pre modelovanja se ponavlja cleaning iz Aktivnosti 3:

- uklanjanje duplikata po kompozitnom ključu,
- filtriranje nevalidne kvadrature, cene, broja soba i godine izgradnje,
- feature engineering za spratnost, starost zgrade i tekstualne indikatore.

`price_per_m2` se izračunava za analizu, ali se ne koristi kao ulaz modela jer direktno zavisi od ciljne promenljive `price_eur`.

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "nekretnine_raw.csv"
raw_df = pd.read_csv(DATA_PATH)
cleaned_df = build_cleaned_dataset(raw_df)
model_df = add_model_features(cleaned_df, current_year=2026)
cleaning_summary = create_cleaning_summary(raw_df)

print(f"Raw skup: {len(raw_df):,} redova".replace(",", "."))
print(f"Model-ready skup: {len(model_df):,} redova".replace(",", "."))
display(pd.DataFrame([cleaning_summary]).T.rename(columns={0: "vrednost"}))
display(model_df.head())

## 2. Train/test podela i preprocessing pipeline

Ciljna promenljiva je `price_eur`. Ulazni atributi su numeričke i kategoričke kolone definisane u `src.features`. Kompletan preprocessing se nalazi u `src.model_pipeline` i obuhvata:

- imputaciju numeričkih vrednosti medianom,
- skaliranje numeričkih atributa,
- imputaciju kategoričkih vrednosti sa `Nepoznato`,
- one-hot encoding kategoričkih atributa.

Podela skupa je 80/20 sa fiksnim `random_state=42`.

In [ ]:
target = "price_eur"
feature_columns = get_model_feature_columns()

X = model_df[feature_columns]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame(
    [
        {"skup": "train", "redovi": len(X_train), "kolone": X_train.shape[1]},
        {"skup": "test", "redovi": len(X_test), "kolone": X_test.shape[1]},
    ]
)

print("Feature kolone korišćene kao ulaz modela:")
print(feature_columns)
display(split_summary)

## Aktivnost 4 - Prva verzija modela / algoritma

Prema zahtevu za Aktivnost 4, ovde se implementira **prvi model** i prikazuje osnovni izlaz sistema.

Prvi model je **LinearRegression** jer je jednostavan, interpretabilan i dobar početni model za regresioni problem. Pored njega se trenira samo jedan baseline: **DummyRegressor** sa medianom. Baseline služi da proverimo da li pravi model uopšte uči korisne obrasce iz podataka.

In [ ]:
baseline_model_key = "dummy_median"
activity_04_model_key = "linear_regression"

activity_04_specs = [
    ("Baseline - DummyRegressor median", baseline_model_key),
    ("LinearRegression", activity_04_model_key),
]

fitted_models = {}
activity_04_results = []

for display_name, model_key in activity_04_specs:
    pipeline = build_model_pipeline(model_key)
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    fitted_models[display_name] = pipeline
    activity_04_results.append(
        evaluate_regression_predictions(display_name, y_test, predictions)
    )

model_results_activity_04 = compare_regression_results(activity_04_results)
display(model_results_activity_04.round(3))

### Osnovni izlaz sistema za Aktivnost 4

Osnovni izlaz sistema je procenjena cena nekretnine. U tabeli se prikazuju stvarna cena, predviđena cena i apsolutna greška za prvu verziju modela.

In [ ]:
linear_predictions = fitted_models["LinearRegression"].predict(X_test)
actual_vs_predicted = pd.DataFrame(
    {
        "actual_price_eur": y_test.values,
        "predicted_price_eur": linear_predictions,
        "absolute_error_eur": np.abs(y_test.values - linear_predictions),
    }
)

print("Primeri predikcija prve verzije modela:")
display(actual_vs_predicted.head(10).round(2))

plot_sample = actual_vs_predicted.sample(
    min(1000, len(actual_vs_predicted)), random_state=RANDOM_STATE
)
plt.figure(figsize=(7, 6))
sns.scatterplot(
    data=plot_sample,
    x="actual_price_eur",
    y="predicted_price_eur",
    alpha=0.35,
    s=18,
)
max_price = max(
    plot_sample["actual_price_eur"].max(),
    plot_sample["predicted_price_eur"].max(),
)
plt.plot([0, max_price], [0, max_price], color="tab:red", linestyle="--", label="Idealno predviđanje")
plt.title("Aktivnost 4 - LinearRegression: stvarna vs. predviđena cena")
plt.xlabel("Stvarna cena (EUR)")
plt.ylabel("Predviđena cena (EUR)")
plt.legend()
plt.tight_layout()

## Aktivnost 5 - Dodatni modeli, evaluacija i status projekta

Prema zahtevu za Aktivnost 5, dodaju se dodatni modeli i sprovodi evaluacija rezultata. Prema temi projekta, potrebno je imati **najmanje tri modela** mašinskog učenja. Zato se u finalnom poređenju koriste:

1. **Linear Regression** - prvi model iz Aktivnosti 4,
2. **Random Forest** - ansambl stabala odlučivanja,
3. **Gradient Boosting** - boosting model za tabularne regresione probleme.

Baseline se ne računa u tri zahtevana ML modela; koristi se samo kao referentna donja granica kvaliteta.

In [ ]:
activity_05_model_specs = [
    ("LinearRegression", "linear_regression"),
    ("RandomForestRegressor", "random_forest"),
    ("GradientBoostingRegressor", "gradient_boosting"),
]

activity_05_results = []

for display_name, model_key in activity_05_model_specs:
    if display_name in fitted_models:
        pipeline = fitted_models[display_name]
    else:
        pipeline = build_model_pipeline(model_key)
        pipeline.fit(X_train, y_train)
        fitted_models[display_name] = pipeline

    predictions = pipeline.predict(X_test)
    activity_05_results.append(
        evaluate_regression_predictions(display_name, y_test, predictions)
    )

model_results_activity_05 = compare_regression_results(activity_05_results)
best_model_name = model_results_activity_05.iloc[0]["model"]

display(model_results_activity_05.round(3))
print(f"Najbolji model prema MAE: {best_model_name}")

### Vizuelno poređenje modela

Modeli se porede kroz MAE, RMSE i R². Za izbor najboljeg modela primarno se koristi MAE jer direktno pokazuje prosečnu apsolutnu grešku u evrima.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics_for_plot = [
    ("mae", "MAE (EUR) - niže je bolje"),
    ("rmse", "RMSE (EUR) - niže je bolje"),
    ("r2", "R² - više je bolje"),
]

for axis, (metric, title) in zip(axes, metrics_for_plot):
    ascending = metric != "r2"
    plot_df = model_results_activity_05.sort_values(metric, ascending=ascending)
    sns.barplot(data=plot_df, x=metric, y="model", ax=axis, color="tab:blue")
    axis.set_title(title)
    axis.set_xlabel(metric.upper())
    axis.set_ylabel("")

plt.tight_layout()

best_predictions = fitted_models[best_model_name].predict(X_test)
best_actual_vs_predicted = pd.DataFrame(
    {
        "actual_price_eur": y_test.values,
        "predicted_price_eur": best_predictions,
        "absolute_error_eur": np.abs(y_test.values - best_predictions),
    }
)

display(best_actual_vs_predicted.describe().round(2))

## Trenutni status projekta

Ovaj odeljak eksplicitno prikazuje trenutni status projekta nakon Aktivnosti 4 i 5.

- Skup podataka je prikupljen web scraping-om i ima više od 1.000 oglasa.
- Eksplorativna analiza i preprocessing su urađeni u `notebooks/analysis.ipynb`.
- Feature engineering i model pipeline su izdvojeni u reusable `src/` kod.
- Aktivnost 4 je pokrivena prvim modelom: **LinearRegression**.
- Aktivnost 5 je pokrivena dodatnim modelima i evaluacijom: **RandomForestRegressor** i **GradientBoostingRegressor**.
- Tema projekta zahteva najmanje tri modela, što je pokriveno kombinacijom: **Linear Regression, Random Forest i Gradient Boosting**.
- Sledeći korak je Streamlit aplikacija za procenu cene nekretnine na osnovu unetih karakteristika.

In [ ]:
status_summary = pd.DataFrame(
    [
        {"stavka": "Očišćen model-ready skup", "status": f"{len(model_df):,} redova".replace(",", ".")},
        {"stavka": "Aktivnost 4 - prvi model", "status": "LinearRegression"},
        {"stavka": "Aktivnost 5 - dodatni modeli", "status": "RandomForestRegressor, GradientBoostingRegressor"},
        {"stavka": "Ukupan broj ML modela", "status": str(len(activity_05_model_specs))},
        {"stavka": "Najbolji trenutni model prema MAE", "status": str(best_model_name)},
        {"stavka": "Streamlit priprema", "status": "Reusable src pipeline spreman za aplikaciju"},
    ]
)
display(status_summary)